Import relevant Python Packages

In [2]:
import numpy as np
import pandas as pd

Create Our Synthetic Homes

In [3]:
n_homes = 50
home_ids = [f"h{i:03}" for i in range(1, n_homes + 1)]

In [4]:
bed_capacity = np.random.choice(
    [30, 40, 50, 60, 70, 80],
    size=n_homes,
    p=[0.1, 0.2, 0.3, 0.2, 0.15, 0.05]
)

In [5]:
homes = pd.DataFrame({
    "home_id": home_ids,
    "bed_capacity": bed_capacity
})

In [6]:
homes["base_occupancy_rate"] = np.random.uniform(0.77, 0.93, size=n_homes)

In [7]:
dates = pd.date_range(start="2026-01-01", end="2026-12-31")
df = homes.merge(pd.DataFrame({"date": dates}), how="cross")

df["occupancy_rate"] = df["base_occupancy_rate"] + np.random.normal(0, 0.07, size=len(df))
df["occupancy_rate"] = df["occupancy_rate"].clip(0.6, 1.0)

In [8]:
df["occupied_beds"] = (df["occupancy_rate"] * df["bed_capacity"]).round().astype(int)

In [9]:
df.head()
df.shape

(18250, 6)

In [10]:
df.head()

,home_id,bed_capacity,base_occupancy_rate,date,occupancy_rate,occupied_beds
0,h001,70,0.911431,2026-01-01,0.881929,62
1,h001,70,0.911431,2026-01-02,0.721556,51
2,h001,70,0.911431,2026-01-03,1.000000,70
3,h001,70,0.911431,2026-01-04,1.000000,70
4,h001,70,0.911431,2026-01-05,0.913140,64


In [11]:
import os
os.makedirs("data", exist_ok=True)

homes.to_csv("data/homes.csv", index=False)
df.to_csv("data/df_temp.csv", index=False)  # temp — will be replaced later

In [12]:
from faker import Faker
from datetime import date
fake = Faker('en_GB')

np.random.seed(42)

DEPENDENCY_HOURS = {1: 2, 2: 3, 3: 4.5, 4: 6, 5: 7, 6: 8}

residents = []

for _, home in homes.iterrows():
    n_starting = round(home["bed_capacity"] * home["base_occupancy_rate"])
    
    for _ in range(n_starting):
        discharges_during_year = np.random.random() < 0.3
        admission_date = date(2026, 1, 1)
        discharge_date = (
            fake.date_between(start_date=date(2026, 2, 1), end_date=date(2026, 12, 31))
            if discharges_during_year else None
        )
        residents.append({
            "resident_id": f"r{len(residents)+1:04}",
            "home_id": home["home_id"],
            "name": fake.name(),
            "dob": fake.date_of_birth(minimum_age=65, maximum_age=100),
            "gender": np.random.choice(["M", "F"], p=[0.4, 0.6]),
            "dependency_score": np.random.choice(
                [1, 2, 3, 4, 5, 6],
                p=[0.05, 0.10, 0.20, 0.30, 0.25, 0.10]
            ),
            "admission_date": admission_date,
            "discharge_date": discharge_date
        })
    
    n_admissions = round(n_starting * 0.3)
    for _ in range(n_admissions):
        admission_date = fake.date_between(start_date=date(2026, 2, 1), end_date=date(2026, 11, 30))
        discharge_date = (
            fake.date_between(start_date=admission_date, end_date=date(2026, 12, 31))
            if np.random.random() < 0.15 else None
        )
        residents.append({
            "resident_id": f"r{len(residents)+1:04}",
            "home_id": home["home_id"],
            "name": fake.name(),
            "dob": fake.date_of_birth(minimum_age=65, maximum_age=100),
            "gender": np.random.choice(["M", "F"], p=[0.4, 0.6]),
            "dependency_score": np.random.choice(
                [1, 2, 3, 4, 5, 6],
                p=[0.05, 0.10, 0.20, 0.30, 0.25, 0.10]
            ),
            "admission_date": pd.Timestamp(admission_date),
            "discharge_date": pd.Timestamp(discharge_date) if discharge_date else None
        })

residents = pd.DataFrame(residents)
residents.to_csv("data/residents.csv", index=False)
residents.head()

,resident_id,home_id,name,dob,gender,dependency_score,admission_date,discharge_date
0,r0001,h001,Joan Brown-Murphy,1934-06-23,F,5,2026-01-01,None
1,r0002,h001,Samuel Jones-Thomson,1938-08-31,M,3,2026-01-01,None
2,r0003,h001,Harry Bray-Newton,1942-10-04,F,4,2026-01-01,2026-12-21
3,r0004,h001,Frank Smith,1947-03-11,M,6,2026-01-01,None
4,r0005,h001,Connor Miah,1946-07-21,M,3,2026-01-01,None


In [13]:
print(f"Total residents: {len(residents)}")
print(f"Homes covered: {residents['home_id'].nunique()}")
print(f"\nResidents per home (sample):")
print(residents.groupby("home_id").size().describe())
print(f"\nDependency distribution:")
print(residents["dependency_score"].value_counts().sort_index())
print(f"\nDischarge rate: {residents['discharge_date'].notna().mean():.1%}")

Total residents: 2809
Homes covered: 50

Residents per home (sample):
count    50.000
mean     56.180
std      14.709
min      31.000
25%      45.000
50%      56.000
75%      68.000
max      95.000
dtype: float64

Dependency distribution:
dependency_score
1    143
2    315
3    571
4    838
5    667
6    275
Name: count, dtype: int64

Discharge rate: 25.5%


In [14]:
dates = pd.date_range(start="2026-01-01", end="2026-12-31")

# convert to datetime for comparison
residents["admission_date"] = pd.to_datetime(residents["admission_date"])
residents["discharge_date"] = pd.to_datetime(residents["discharge_date"])

census_rows = []

for date in dates:
    active = residents[
        (residents["admission_date"] <= date) &
        (residents["discharge_date"].isna() | (residents["discharge_date"] >= date))
    ]
    daily = active.groupby("home_id").agg(
        occupied_beds=("resident_id", "count"),
        avg_dependency=("dependency_score", "mean")
    ).reset_index()
    daily["date"] = date
    census_rows.append(daily)

daily_census = pd.concat(census_rows, ignore_index=True)
daily_census = daily_census.merge(homes[["home_id", "bed_capacity"]], on="home_id")
daily_census["occupancy_rate"] = daily_census["occupied_beds"] / daily_census["bed_capacity"]

daily_census.to_csv("data/daily_census.csv", index=False)
daily_census.head()

,home_id,occupied_beds,avg_dependency,date,bed_capacity,occupancy_rate
0,h001,64,3.828125,2026-01-01,70,0.914286
1,h002,28,3.750000,2026-01-01,30,0.933333
2,h003,33,3.575758,2026-01-01,40,0.825000
3,h004,34,4.088235,2026-01-01,40,0.850000
4,h005,37,3.729730,2026-01-01,40,0.925000


In [15]:
print(f"Shape: {daily_census.shape}")
print(f"\nOccupancy rate stats:")
print(daily_census["occupancy_rate"].describe())
print(f"\nAvg dependency stats:")
print(daily_census["avg_dependency"].describe())
print(f"\nSample — home h001:")
print(daily_census[daily_census["home_id"] == "h001"].head(10))

Shape: (18250, 6)

Occupancy rate stats:
count    18250.000000
mean         0.879002
std          0.073546
min          0.650000
25%          0.833333
50%          0.883333
75%          0.928571
max          1.050000
Name: occupancy_rate, dtype: float64

Avg dependency stats:
count    18250.000000
mean         3.851364
std          0.187672
min          3.363636
25%          3.708333
50%          3.823529
75%          4.000000
max          4.404762
Name: avg_dependency, dtype: float64

Sample — home h001:
    home_id  occupied_beds  avg_dependency       date  bed_capacity  \
0      h001             64        3.828125 2026-01-01            70   
50     h001             64        3.828125 2026-01-02            70   
100    h001             64        3.828125 2026-01-03            70   
150    h001             64        3.828125 2026-01-04            70   
200    h001             64        3.828125 2026-01-05            70   
250    h001             64        3.828125 2026-01-06          

In [16]:
daily_census["occupancy_rate"] = daily_census["occupancy_rate"].clip(upper=1.0)
daily_census.to_csv("data/daily_census.csv", index=False)